# EfficientNetB0 Lottery Ticket Experiment

This notebook applies the **Lottery Ticket Hypothesis (LTH)** to the resized 224x224
nits-vs-other dataset.

It runs two initialization modes:

* **pretrained** - EfficientNetB0 starts from ImageNet weights plus a random head, matching
  the transfer-learning starting point used by `training_resized.ipynb`.
* **random** - EfficientNetB0 starts from random weights and is trained from scratch, which
  is the faithful LTH setup from `small.md`.

The procedure is iterative magnitude pruning with weight rewinding:

1. Save the initial weights.
2. Train the unpruned network.
3. Prune the smallest-magnitude kernels to the next target sparsity.
4. Rewind the remaining weights to their saved initial values.
5. Reapply the mask and retrain.

Pruning stops when **validation F1** drops more than 0.01 below the unpruned baseline, or at
95% sparsity. Test and comb-OOD metrics are only reported afterward, never used for stopping.

The notebook also exports and compares:

* dense `.keras` checkpoints,
* sparse `.npz` weight files,
* float16 and dynamic-range TFLite models,

and finishes with a direct comparison against the already-trained
`resized/models/EfficientNetB0.keras`.

All outputs go to `resized/lottery_ticket/`.


## 1. Environment & configuration

In [ ]:
import os, json, time, ctypes, glob, sysconfig, gc
from pathlib import Path

# --- Preload the NVIDIA pip libraries -------------------------------------
# tensorflow[and-cuda] installs CUDA/cuDNN under site-packages/nvidia, but the
# dynamic loader does not search there by default. Loading them with
# RTLD_GLOBAL before importing TensorFlow makes the GPU visible without
# requiring a manually exported LD_LIBRARY_PATH.
_nvidia_root = Path(sysconfig.get_paths()["purelib"]) / "nvidia"
_NVIDIA_LIB_PATTERNS = [
    "cuda_runtime/lib/libcudart.so*",
    "cublas/lib/libcublasLt.so*", "cublas/lib/libcublas.so*",
    "cudnn/lib/libcudnn.so*", "cudnn/lib/libcudnn_ops.so*",
    "cudnn/lib/libcudnn_cnn.so*", "cudnn/lib/libcudnn_adv.so*",
    "cudnn/lib/libcudnn_graph.so*", "cudnn/lib/libcudnn_engines_*.so*",
    "cudnn/lib/libcudnn_heuristic.so*",
    "cufft/lib/libcufft.so*", "curand/lib/libcurand.so*",
    "cusolver/lib/libcusolver.so*", "cusparse/lib/libcusparse.so*",
    "nccl/lib/libnccl.so*", "nvjitlink/lib/libnvJitLink.so*",
    "cuda_nvrtc/lib/libnvrtc.so*",
]
if _nvidia_root.exists():
    for _pattern in _NVIDIA_LIB_PATTERNS:
        for _lib in sorted(glob.glob(str(_nvidia_root / _pattern))):
            try:
                ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from IPython.display import display
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_curve, auc

print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("CPUs:", os.cpu_count())

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# --- GPU setup ---
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print("GPU memory-growth warning:", exc)
HAS_GPU = len(gpus) > 0
print("GPU enabled" if HAS_GPU else "No GPU detected - using CPU")

# LTH masks are easier to reason about in float32. The GPU is still used.
tf.keras.mixed_precision.set_global_policy("float32")

# --- Config ---
IMG_SIZE = 224
BATCH_SIZE = 64 if HAS_GPU else 32
AUTOTUNE = tf.data.AUTOTUNE
THRESHOLD = 0.5

DATA_DIR = Path("resized")
SPLITS_PATH = DATA_DIR / "results" / "splits.json"
PRODUCTION_MODEL = DATA_DIR / "models" / "EfficientNetB0.keras"
COMB_EVAL_CSV = DATA_DIR / "results" / "comb_nits_eval_set.csv"
OUT_ROOT = DATA_DIR / "lottery_ticket"

INIT_MODES = ("pretrained", "random")
SPARSITY_TARGETS = (0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95)
MAX_F1_DROP = 0.01
BASELINE_EPOCHS = {"pretrained": 20, "random": 40}
ROUND_EPOCHS = {"pretrained": 10, "random": 15}
EARLY_STOP_PATIENCE = {"pretrained": 5, "random": 8}
LEARNING_RATE = {"pretrained": 1e-4, "random": 1e-3}

for mode in INIT_MODES:
    (OUT_ROOT / mode).mkdir(parents=True, exist_ok=True)

print("Output root:", OUT_ROOT)
print("Init modes:", INIT_MODES)
print("Sparsity targets:", SPARSITY_TARGETS)
print("Stop rule: validation F1 < baseline -", MAX_F1_DROP)


## 2. Load the shared split and build the data pipeline

In [ ]:
splits = json.loads(SPLITS_PATH.read_text())

def label_from_path(path):
    parts = Path(path).parts
    return int("nits" in parts and "nits on comb images" not in parts)

def make_split_df(paths):
    paths = [p for p in paths if Path(p).exists()]
    y = np.array([label_from_path(p) for p in paths], dtype=np.int64)
    return pd.DataFrame({
        "filepath": paths,
        "y": y,
        "label": np.where(y == 1, "nits", "other"),
    })

train_df = make_split_df(splits["train"])
val_df = make_split_df(splits["val"])
test_df = make_split_df(splits["test"])

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name:5s}", d["label"].value_counts().to_dict())

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["y"].values)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print("class_weight:", class_weight)

EFFICIENTNET_PREPROCESS = tf.keras.applications.efficientnet.preprocess_input

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")

def decode_img(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.ensure_shape(img, [None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)

def make_ds(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_img, num_parallel_calls=AUTOTUNE)
    # Only cache the small validation/test sets. Caching the augmented training
    # set keeps ~1 GB of decoded images resident for the whole notebook.
    if not training:
        ds = ds.cache()
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (EFFICIENTNET_PREPROCESS(x), y),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_df["filepath"].values, train_df["y"].values, True)
val_ds = make_ds(val_df["filepath"].values, val_df["y"].values, False)
test_ds = make_ds(test_df["filepath"].values, test_df["y"].values, False)
print("datasets ready")


## 3. Model and mask utilities

In [ ]:
def build_efficientnet(init_mode):
    weights = "imagenet" if init_mode == "pretrained" else None
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    # The fixed ImageNet normalisation constants are preprocessing, not learned
    # parameters. Keep them for the random-init run too.
    if init_mode == "random":
        for layer in base.layers:
            if isinstance(layer, tf.keras.layers.Normalization):
                layer.weights[0].assign(np.array([0.485, 0.456, 0.406], dtype=np.float32))
                layer.weights[1].assign(np.array([0.229, 0.224, 0.225], dtype=np.float32))
                layer.weights[2].assign(0.0)

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return tf.keras.Model(inputs, outputs, name=f"EfficientNetB0_{init_mode}")

def make_full_masks(model):
    return [tf.ones(w.shape, dtype=tf.bool) if len(w.shape) >= 2 else None
            for w in model.trainable_weights]

def apply_masks(model, masks):
    for w, m in zip(model.trainable_weights, masks):
        if m is not None:
            w.assign(w * tf.cast(m, w.dtype))

def count_pruneable(model):
    return sum(int(np.prod(w.shape)) for w in model.trainable_weights if len(w.shape) >= 2)

def sparsity_of(model, masks):
    total = sum(int(np.prod(w.shape)) for w, m in zip(model.trainable_weights, masks)
                if m is not None)
    kept = sum(int(tf.reduce_sum(tf.cast(m, tf.int32))) for m in masks if m is not None)
    return 1.0 - kept / max(total, 1)

def prune_masks(model, masks, target_sparsity):
    total_pruneable = sum(int(np.prod(w.shape)) for w, m in zip(model.trainable_weights, masks)
                          if m is not None)
    total_keep = int(round((1.0 - target_sparsity) * total_pruneable))

    active_values = []
    for w, m in zip(model.trainable_weights, masks):
        if m is None:
            continue
        active_values.append(tf.boolean_mask(tf.abs(w), tf.cast(m, tf.bool)))
    active_values = tf.concat(active_values, axis=0)

    if total_keep <= 0:
        threshold = tf.constant(np.inf, dtype=active_values.dtype)
    elif total_keep >= int(active_values.shape[0]):
        threshold = tf.constant(-1.0, dtype=active_values.dtype)
    else:
        threshold = tf.math.top_k(active_values, k=total_keep).values[-1]

    new_masks = []
    for w, m in zip(model.trainable_weights, masks):
        if m is None:
            new_masks.append(None)
        else:
            new_masks.append(tf.logical_and(tf.cast(m, tf.bool), tf.abs(w) >= threshold))
    return new_masks

def nonzero_param_count(model, masks=None):
    total = 0
    if masks is None:
        total = int(model.count_params())
    else:
        for w, m in zip(model.trainable_weights, masks):
            if m is None:
                total += int(np.prod(w.shape))
            else:
                total += int(tf.reduce_sum(tf.cast(m, tf.int32)))
        total += sum(int(np.prod(w.shape)) for w in model.non_trainable_weights)
    return total

def weights_only_mb(model, masks=None):
    return nonzero_param_count(model, masks) * 4 / 1e6

def per_layer_sparsity(model, masks):
    rows = []
    for i, (w, m) in enumerate(zip(model.trainable_weights, masks)):
        if m is None:
            continue
        total = int(np.prod(w.shape))
        kept = int(tf.reduce_sum(tf.cast(m, tf.int32)))
        rows.append(dict(index=i, name=w.name, shape=str(tuple(w.shape)),
                         total=total, kept=kept,
                         sparsity=1.0 - kept / max(total, 1)))
    return pd.DataFrame(rows)


## 4. Masked training and evaluation

In [ ]:
def make_train_step(model, masks, optimizer):
    @tf.function
    def train_step(x, y):
        y = tf.cast(y, tf.float32)
        with tf.GradientTape() as tape:
            y_pred = tf.cast(model(x, training=True), tf.float32)
            bce = tf.keras.losses.binary_crossentropy(tf.reshape(y, (-1, 1)), y_pred)
            sample_weight = tf.where(tf.equal(y, 1.0), class_weight[1], class_weight[0])
            loss = tf.reduce_mean(bce * sample_weight)

        grads = tape.gradient(loss, model.trainable_weights)
        masked_grads = []
        for g, m in zip(grads, masks):
            if g is None:
                masked_grads.append(None)
            elif m is None:
                masked_grads.append(g)
            else:
                masked_grads.append(g * tf.cast(m, g.dtype))

        optimizer.apply_gradients(zip(masked_grads, model.trainable_weights))
        for w, m in zip(model.trainable_weights, masks):
            if m is not None:
                w.assign(w * tf.cast(m, w.dtype))
        return loss
    return train_step

def evaluate_loader(model, ds):
    loss_m = tf.keras.metrics.Mean()
    acc = tf.keras.metrics.BinaryAccuracy()
    prec = tf.keras.metrics.Precision()
    rec = tf.keras.metrics.Recall()
    auc_m = tf.keras.metrics.AUC()

    for x, y in ds:
        y = tf.cast(y, tf.float32)
        y2 = tf.reshape(y, (-1, 1))
        p = tf.cast(model(x, training=False), tf.float32)
        loss_m.update_state(tf.keras.losses.binary_crossentropy(y2, p))
        acc.update_state(y2, p)
        prec.update_state(y2, p)
        rec.update_state(y2, p)
        auc_m.update_state(y2, p)

    pr = float(prec.result())
    rc = float(rec.result())
    return dict(loss=float(loss_m.result()), acc=float(acc.result()),
                precision=pr, recall=rc,
                f1=2 * pr * rc / (pr + rc + 1e-8),
                auc=float(auc_m.result()))

def train_ticket(model, masks, epochs, lr, patience, verbose=True):
    optimizer = tf.keras.optimizers.Adam(lr, clipnorm=1.0)
    step = make_train_step(model, masks, optimizer)
    best_auc = -np.inf
    best_weights = None
    best_metrics = None
    wait = 0
    history = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        losses = []
        for x, y in train_ds:
            losses.append(float(step(x, y)))
        val = evaluate_loader(model, val_ds)
        row = dict(epoch=epoch, train_loss=float(np.mean(losses)),
                   **{f"val_{k}": v for k, v in val.items()},
                   seconds=round(time.time() - t0, 1))
        history.append(row)
        if verbose:
            print(f"    epoch {epoch:02d}/{epochs} loss={row['train_loss']:.4f} "
                  f"val_f1={val['f1']:.4f} val_auc={val['auc']:.4f} ({row['seconds']}s)")

        if val["auc"] > best_auc + 1e-6:
            best_auc = val["auc"]
            best_weights = [w.copy() for w in model.get_weights()]
            best_metrics = val
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"    early stop at epoch {epoch}")
                break

    if best_weights is not None:
        model.set_weights(best_weights)
        apply_masks(model, masks)
    return pd.DataFrame(history), best_metrics


## 5. Test and comb-OOD evaluation helpers

In [ ]:
def predict_test(model):
    y_prob = model.predict(test_ds, verbose=0).ravel()
    y_true = test_df["y"].values
    y_pred = (y_prob >= THRESHOLD).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    metrics = dict(
        acc=(tp + tn) / len(y_true),
        precision=precision,
        recall=recall,
        f1=2 * precision * recall / (precision + recall + 1e-8),
        auc=float(__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y_true, y_prob)),
    )
    pred_df = pd.DataFrame({
        "filepath": test_df["filepath"].values,
        "true_label": test_df["label"].values,
        "y_true": y_true,
        "prob_nits": y_prob,
        "predicted_label": np.where(y_pred == 1, "nits", "other"),
        "confidence_pct": 100.0 * np.maximum(y_prob, 1.0 - y_prob),
        "correct": y_pred == y_true,
    })
    return metrics, pred_df

def decode_img_mode(path, mode="squash"):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.ensure_shape(img, [None, None, 3])
    if mode == "squash":
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    elif mode == "centercrop":
        h, w = tf.shape(img)[0], tf.shape(img)[1]
        s = tf.minimum(h, w)
        img = tf.image.crop_to_bounding_box(img, (h - s) // 2, (w - s) // 2, s, s)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    elif mode == "letterbox":
        img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE)
    else:
        raise ValueError(f"unknown mode: {mode}")
    return tf.cast(img, tf.float32)

def predict_paths(model, paths, mode="squash"):
    ds = tf.data.Dataset.from_tensor_slices(list(paths))
    ds = ds.map(lambda p: decode_img_mode(p, mode), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).cache()
    ds = ds.map(lambda x: EFFICIENTNET_PREPROCESS(x), num_parallel_calls=AUTOTUNE)
    return model.predict(ds.prefetch(AUTOTUNE), verbose=0).ravel()

if COMB_EVAL_CSV.exists():
    comb_eval_df = pd.read_csv(COMB_EVAL_CSV)
else:
    raise FileNotFoundError(
        f"{COMB_EVAL_CSV} not found. Run training_resized.ipynb first so the comb "
        "evaluation set is available.")
print("comb evaluation set:", len(comb_eval_df), "images")

def evaluate_comb(model, modes=("squash", "centercrop", "letterbox")):
    y_true = comb_eval_df["y"].values
    rows = []
    probs_all = {}
    per_image = comb_eval_df.copy()

    for mode in modes:
        probs = predict_paths(model, comb_eval_df["filepath"].tolist(), mode)
        probs_all[mode] = probs
        if mode == "squash":
            per_image["prob_nits"] = probs
        y_pred = (probs >= THRESHOLD).astype(int)
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        rows.append(dict(
            resize=mode,
            nits_detected=tp, nits_total=tp + fn,
            detect_pct=100.0 * tp / max(tp + fn, 1),
            false_nits=fp, other_total=fp + tn,
            false_pct=100.0 * fp / max(fp + tn, 1),
            accuracy=(tp + tn) / len(y_true),
            precision=tp / (tp + fp) if (tp + fp) else 0.0,
            recall=tp / (tp + fn) if (tp + fn) else 0.0,
            f1=(2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else 0.0,
            mean_prob_nits=float(probs[y_true == 1].mean()),
            mean_prob_other=float(probs[y_true == 0].mean()),
        ))
    return pd.DataFrame(rows), per_image, probs_all

def evaluate_all(model):
    test_metrics, test_pred = predict_test(model)
    comb_summary, comb_per_image, comb_probs = evaluate_comb(model)
    return test_metrics, test_pred, comb_summary, comb_per_image, comb_probs


## 6. Export and size helpers

In [ ]:
def file_mb(path):
    p = Path(path)
    return p.stat().st_size / 1e6 if p.exists() else float("nan")

def save_sparse_weights(model, masks, path):
    arrays = {}
    nonzero = 0
    for i, (w, m) in enumerate(zip(model.trainable_weights, masks)):
        if m is None:
            continue
        values = w.numpy()
        nz = np.nonzero(values)
        arrays[f"indices_{i}"] = np.stack(nz, axis=1).astype(np.int32)
        arrays[f"values_{i}"] = values[nz].astype(np.float32)
        arrays[f"shape_{i}"] = np.array(values.shape, dtype=np.int32)
        nonzero += len(values[nz])
    arrays["nonzero_count"] = np.array([nonzero], dtype=np.int64)
    np.savez_compressed(path, **arrays)
    return file_mb(path)

def export_tflite(model, path, float16=False):
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        if float16:
            converter.target_spec.supported_types = [tf.float16]
        tflite_model = converter.convert()
        Path(path).write_bytes(tflite_model)
        return file_mb(path)
    except Exception as exc:
        Path(str(path) + ".error.txt").write_text(f"{type(exc).__name__}: {exc}")
        print(f"  TFLite export failed for {path}: {type(exc).__name__}: {exc}")
        return float("nan")
    finally:
        gc.collect()

def measure_latency(model, n=32, repeats=5):
    x_batch, _ = next(iter(test_ds))
    x_batch = x_batch[:n]
    for _ in range(2):
        _ = np.asarray(model(x_batch, training=False))
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        _ = np.asarray(model(x_batch, training=False))
        times.append(time.perf_counter() - t0)
    return 1000.0 * float(np.median(times)) / len(x_batch)


## 7. Run LTH for both initialization modes

In [ ]:
def run_lth(init_mode):
    print(f"\n{'#' * 72}\n# LTH: {init_mode} initialization\n{'#' * 72}")
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.mixed_precision.set_global_policy("float32")
    tf.keras.utils.set_random_seed(SEED)

    out_dir = OUT_ROOT / init_mode
    out_dir.mkdir(parents=True, exist_ok=True)

    model = build_efficientnet(init_mode)
    model.save_weights(str(out_dir / "initial_weights.weights.h5"))
    initial_weights = [w.copy() for w in model.get_weights()]
    masks = make_full_masks(model)
    pruneable_total = count_pruneable(model)
    print(f"params={model.count_params():,} | pruneable={pruneable_total:,} | "
          f"initial weights saved to {out_dir / 'initial_weights.weights.h5'}")

    rows = []

    def record(tag, stage, target, model, masks, val_metrics, test_metrics,
               comb_summary, artifact_paths, latency_ms):
        sq = comb_summary[comb_summary["resize"] == "squash"].iloc[0]
        row = dict(
            init=init_mode, stage=stage, tag=tag, target_sparsity=target,
            actual_sparsity=sparsity_of(model, masks),
            val_f1=val_metrics["f1"], val_auc=val_metrics["auc"],
            test_acc=test_metrics["acc"], test_precision=test_metrics["precision"],
            test_recall=test_metrics["recall"], test_f1=test_metrics["f1"],
            test_auc=test_metrics["auc"],
            comb_detect_pct=sq["detect_pct"], comb_false_pct=sq["false_pct"],
            comb_f1=sq["f1"], latency_ms=latency_ms,
            params=int(model.count_params()),
            nonzero_params=nonzero_param_count(model, masks),
            effective_mb=weights_only_mb(model, masks),
            **artifact_paths,
        )
        rows.append(row)
        pd.DataFrame(rows).to_csv(out_dir / "lth_history.csv", index=False)
        return row

    print("\nTraining unpruned baseline ...")
    hist, val_metrics = train_ticket(
        model, masks, BASELINE_EPOCHS[init_mode],
        LEARNING_RATE[init_mode], EARLY_STOP_PATIENCE[init_mode])
    hist.to_csv(out_dir / "baseline_history.csv", index=False)
    model.save(str(out_dir / "baseline.keras"))

    test_metrics, test_pred, comb_summary, comb_per, comb_probs = evaluate_all(model)
    test_pred.to_csv(out_dir / "baseline_test_predictions.csv", index=False)
    comb_summary.to_csv(out_dir / "baseline_comb_summary.csv", index=False)
    per_layer_sparsity(model, masks).to_csv(out_dir / "baseline_layer_sparsity.csv", index=False)

    tflite_f16 = export_tflite(model, out_dir / "baseline_float16.tflite", float16=True)
    tflite_dyn = export_tflite(model, out_dir / "baseline_dynamic.tflite", float16=False)
    latency = measure_latency(model)
    baseline_row = record(
        "baseline", "baseline", 0.0, model, masks, val_metrics, test_metrics,
        comb_summary,
        dict(keras_mb=file_mb(out_dir / "baseline.keras"), sparse_mb=np.nan,
             tflite_f16_mb=tflite_f16, tflite_dyn_mb=tflite_dyn),
        latency)
    baseline_val_f1 = val_metrics["f1"]
    print(f"baseline: val_f1={baseline_val_f1:.4f} test_f1={test_metrics['f1']:.4f} "
          f"comb_detect={baseline_row['comb_detect_pct']:.1f}%")

    accepted = []
    for target in SPARSITY_TARGETS:
        print(f"\n--- target sparsity {target:.0%} ---")
        new_masks = prune_masks(model, masks, target)
        model.set_weights(initial_weights)
        apply_masks(model, new_masks)
        print(f"  rewound to initial weights; current sparsity="
              f"{sparsity_of(model, new_masks):.3f}")

        hist, val_metrics = train_ticket(
            model, new_masks, ROUND_EPOCHS[init_mode],
            LEARNING_RATE[init_mode], EARLY_STOP_PATIENCE[init_mode])
        actual_sparsity = sparsity_of(model, new_masks)
        tag = f"ticket_{int(round(actual_sparsity * 100)):02d}"
        hist.to_csv(out_dir / f"{tag}_history.csv", index=False)
        model.save(str(out_dir / f"{tag}.keras"))
        per_layer_sparsity(model, new_masks).to_csv(
            out_dir / f"{tag}_layer_sparsity.csv", index=False)

        test_metrics, test_pred, comb_summary, comb_per, comb_probs = evaluate_all(model)
        test_pred.to_csv(out_dir / f"{tag}_test_predictions.csv", index=False)
        comb_summary.to_csv(out_dir / f"{tag}_comb_summary.csv", index=False)

        sparse_mb = save_sparse_weights(model, new_masks, out_dir / f"{tag}_sparse.npz")
        # TFLite is exported only for the selected tickets later. Converting every
        # round leaks converter memory, and unstructured zeros do not shrink the
        # TFLite file anyway.
        tflite_f16 = np.nan
        tflite_dyn = np.nan
        latency = measure_latency(model)
        gc.collect()

        row = record(
            tag, "ticket", target, model, new_masks, val_metrics, test_metrics,
            comb_summary,
            dict(keras_mb=file_mb(out_dir / f"{tag}.keras"), sparse_mb=sparse_mb,
                 tflite_f16_mb=tflite_f16, tflite_dyn_mb=tflite_dyn),
            latency)
        print(f"  actual_sparsity={actual_sparsity:.3f} val_f1={val_metrics['f1']:.4f} "
              f"test_f1={test_metrics['f1']:.4f} comb_detect={row['comb_detect_pct']:.1f}% "
              f"effective={row['effective_mb']:.2f}MB tflite_dyn={row['tflite_dyn_mb']:.2f}MB")

        if val_metrics["f1"] < baseline_val_f1 - MAX_F1_DROP:
            print(f"  STOP: validation F1 dropped below baseline - {MAX_F1_DROP:.2f} "
                  f"({baseline_val_f1:.4f} -> {val_metrics['f1']:.4f})")
            break

        accepted.append(row)
        masks = new_masks

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "lth_history.csv", index=False)

    if accepted:
        acc_df = pd.DataFrame(accepted)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(acc_df["actual_sparsity"], acc_df["val_f1"], marker="o", label="val F1")
        axes[0].plot(acc_df["actual_sparsity"], acc_df["test_f1"], marker="s", label="test F1")
        axes[0].axhline(baseline_val_f1, color="gray", ls="--", label="baseline val F1")
        axes[0].set_xlabel("sparsity"); axes[0].set_ylabel("F1")
        axes[0].set_title(f"{init_mode}: F1 vs sparsity"); axes[0].legend()

        axes[1].plot(acc_df["actual_sparsity"], acc_df["comb_detect_pct"],
                     marker="o", label="comb detect %")
        axes[1].plot(acc_df["actual_sparsity"], acc_df["comb_false_pct"],
                     marker="s", label="comb false %")
        axes[1].set_xlabel("sparsity"); axes[1].set_ylabel("%")
        axes[1].set_title(f"{init_mode}: comb OOD"); axes[1].legend()

        axes[2].plot(acc_df["effective_mb"], acc_df["test_f1"], marker="o",
                     label="effective weights size")
        axes[2].plot(acc_df["sparse_mb"], acc_df["test_f1"], marker="s",
                     label="sparse .npz file")
        axes[2].set_xlabel("size (MB)"); axes[2].set_ylabel("test F1")
        axes[2].set_title(f"{init_mode}: size vs F1"); axes[2].legend()

        plt.tight_layout()
        plt.savefig(out_dir / "lth_curves.png", dpi=120)
        plt.show()

    return df, accepted


In [ ]:
all_results = []
all_tickets = {}
for mode in INIT_MODES:
    df, accepted = run_lth(mode)
    all_results.append(df)
    all_tickets[mode] = accepted

combined = pd.concat(all_results, ignore_index=True)
combined.to_csv(OUT_ROOT / "lth_history_all.csv", index=False)
display(combined)


## 8. Compare against the trained resized EfficientNetB0

This section loads `resized/models/EfficientNetB0.keras` and evaluates it on exactly the same
test split and comb evaluation set, then compares it with both baselines and every accepted
ticket.


In [ ]:
tf.keras.backend.clear_session()
gc.collect()
tf.keras.mixed_precision.set_global_policy("float32")

production = tf.keras.models.load_model(PRODUCTION_MODEL, compile=False)
prod_test, prod_test_pred, prod_comb, prod_comb_per, prod_comb_probs = evaluate_all(production)

prod_test_pred.to_csv(OUT_ROOT / "production_test_predictions.csv", index=False)
prod_comb.to_csv(OUT_ROOT / "production_comb_summary.csv", index=False)

# Re-save without optimizer state so the .keras size comparison is fair.
production.save(str(OUT_ROOT / "production_reloaded.keras"))
prod_keras_mb = file_mb(OUT_ROOT / "production_reloaded.keras")
prod_original_keras_mb = file_mb(PRODUCTION_MODEL)
prod_tflite_f16 = export_tflite(production, OUT_ROOT / "production_float16.tflite", float16=True)
prod_tflite_dyn = export_tflite(production, OUT_ROOT / "production_dynamic.tflite", float16=False)
prod_latency = measure_latency(production)

prod_sq = prod_comb[prod_comb["resize"] == "squash"].iloc[0]
prod_row = dict(
    init="production", stage="production", tag="production", target_sparsity=0.0,
    actual_sparsity=0.0, val_f1=np.nan, val_auc=np.nan,
    test_acc=prod_test["acc"], test_precision=prod_test["precision"],
    test_recall=prod_test["recall"], test_f1=prod_test["f1"], test_auc=prod_test["auc"],
    comb_detect_pct=prod_sq["detect_pct"], comb_false_pct=prod_sq["false_pct"],
    comb_f1=prod_sq["f1"], latency_ms=prod_latency,
    params=int(production.count_params()),
    nonzero_params=int(production.count_params()),
    effective_mb=production.count_params() * 4 / 1e6,
    keras_mb=prod_keras_mb, sparse_mb=np.nan,
    tflite_f16_mb=prod_tflite_f16, tflite_dyn_mb=prod_tflite_dyn,
    original_keras_mb=prod_original_keras_mb,
)

comparison = pd.concat([pd.DataFrame([prod_row]), combined], ignore_index=True)
comparison["f1_delta_vs_production"] = comparison["test_f1"] - prod_test["f1"]
comparison["auc_delta_vs_production"] = comparison["test_auc"] - prod_test["auc"]
comparison["effective_size_reduction_pct"] = (
    1.0 - comparison["effective_mb"] / prod_row["effective_mb"]) * 100.0
comparison["tflite_size_reduction_pct"] = (
    1.0 - comparison["tflite_dyn_mb"] / prod_row["tflite_dyn_mb"]) * 100.0
comparison.to_csv(OUT_ROOT / "lth_vs_resized_efficientnet.csv", index=False)

# TFLite is exported only for the best validation and best test tickets. This
# keeps memory bounded while still reporting a real deployable file size.
def export_ticket_tflite(row):
    keras_path = OUT_ROOT / row["init"] / f"{row['tag']}.keras"
    ticket_model = tf.keras.models.load_model(keras_path, compile=False)
    f16 = export_tflite(ticket_model, OUT_ROOT / f"{row['init']}_{row['tag']}_float16.tflite",
                        float16=True)
    dyn = export_tflite(ticket_model, OUT_ROOT / f"{row['init']}_{row['tag']}_dynamic.tflite",
                        float16=False)
    del ticket_model
    gc.collect()
    return f16, dyn

ticket_rows = comparison[comparison["stage"] == "ticket"]
candidates = []
if not ticket_rows.empty:
    candidates.append(ticket_rows.sort_values("val_f1", ascending=False).iloc[0])
    candidates.append(ticket_rows.sort_values("test_f1", ascending=False).iloc[0])

seen_tickets = set()
for cand in candidates:
    key = (cand["init"], cand["tag"])
    if key in seen_tickets:
        continue
    seen_tickets.add(key)
    f16, dyn = export_ticket_tflite(cand)
    mask = (comparison["init"] == cand["init"]) & (comparison["tag"] == cand["tag"])
    comparison.loc[mask, "tflite_f16_mb"] = f16
    comparison.loc[mask, "tflite_dyn_mb"] = dyn

comparison["tflite_size_reduction_pct"] = (
    1.0 - comparison["tflite_dyn_mb"] / prod_row["tflite_dyn_mb"]) * 100.0
comparison.to_csv(OUT_ROOT / "lth_vs_resized_efficientnet.csv", index=False)

(OUT_ROOT / "production_eval.json").write_text(json.dumps(dict(
    test=prod_test, comb=prod_comb.to_dict(orient="records"),
    keras_mb=prod_keras_mb, original_keras_mb=prod_original_keras_mb,
    tflite_f16_mb=prod_tflite_f16, tflite_dyn_mb=prod_tflite_dyn,
    latency_ms=prod_latency), indent=2))

display(comparison[[
    "init", "stage", "tag", "actual_sparsity", "val_f1", "test_f1", "test_auc",
    "comb_detect_pct", "comb_false_pct", "effective_mb", "tflite_dyn_mb",
    "f1_delta_vs_production", "tflite_size_reduction_pct"
]].sort_values(["stage", "init", "actual_sparsity"]))


## 9. Final comparison plots and verdict

In [ ]:
ticket_df = comparison[comparison["stage"] == "ticket"].copy()
baseline_df = comparison[comparison["stage"] == "baseline"].copy()

best_by_val = ticket_df.sort_values("val_f1", ascending=False).iloc[0] if not ticket_df.empty else None
best_by_test = ticket_df.sort_values("test_f1", ascending=False).iloc[0] if not ticket_df.empty else None

# Rows to show in the summary plots
summary_rows = [prod_row]
for _, r in baseline_df.iterrows():
    summary_rows.append(r.to_dict())
if best_by_val is not None:
    summary_rows.append(best_by_val.to_dict())
if best_by_test is not None and (
        best_by_val is None
        or (best_by_test["init"], best_by_test["tag"]) != (best_by_val["init"], best_by_val["tag"])):
    summary_rows.append(best_by_test.to_dict())
summary_df = pd.DataFrame(summary_rows)
labels = [f"{r['init']}\n{r['tag']}" for _, r in summary_df.iterrows()]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

x = np.arange(len(summary_df))
width = 0.38
axes[0, 0].bar(x - width / 2, summary_df["test_f1"], width, label="test F1")
axes[0, 0].bar(x + width / 2, summary_df["test_auc"], width, label="test AUC")
axes[0, 0].set_xticks(x); axes[0, 0].set_xticklabels(labels, fontsize=8)
axes[0, 0].set_ylim(0.8, 1.02); axes[0, 0].set_title("Held-out test metrics")
axes[0, 0].legend()

axes[0, 1].bar(x - width / 2, summary_df["comb_detect_pct"], width, label="comb detect %")
axes[0, 1].bar(x + width / 2, summary_df["comb_false_pct"], width, label="comb false %")
axes[0, 1].set_xticks(x); axes[0, 1].set_xticklabels(labels, fontsize=8)
axes[0, 1].set_title("Comb OOD metrics"); axes[0, 1].legend()

for _, r in comparison[comparison["stage"].isin(["baseline", "ticket"])].iterrows():
    axes[1, 0].scatter(r["effective_mb"], r["test_f1"],
                       c="#4C72B0" if r["init"] == "pretrained" else "#DD8452",
                       marker="o" if r["stage"] == "ticket" else "s",
                       alpha=0.75, s=55)
axes[1, 0].scatter(prod_row["effective_mb"], prod_row["test_f1"],
                   c="crimson", marker="*", s=260, label="production")
axes[1, 0].set_xlabel("effective weights size (MB)")
axes[1, 0].set_ylabel("held-out test F1")
axes[1, 0].set_title("Size vs accuracy")
axes[1, 0].legend()

# ROC overlay: production + best validation ticket + best test ticket
def add_roc(ax, pred_df, label):
    fpr, tpr, _ = roc_curve(pred_df["y_true"], pred_df["prob_nits"])
    ax.plot(fpr, tpr, label=f"{label} (AUC={auc(fpr, tpr):.3f})")

add_roc(axes[1, 1], prod_test_pred, "production")
if best_by_val is not None:
    p = OUT_ROOT / best_by_val["init"] / f"{best_by_val['tag']}_test_predictions.csv"
    if p.exists():
        add_roc(axes[1, 1], pd.read_csv(p), f"best val {best_by_val['init']}/{best_by_val['tag']}")
if best_by_test is not None:
    p = OUT_ROOT / best_by_test["init"] / f"{best_by_test['tag']}_test_predictions.csv"
    if p.exists():
        add_roc(axes[1, 1], pd.read_csv(p), f"best test {best_by_test['init']}/{best_by_test['tag']}")
axes[1, 1].plot([0, 1], [0, 1], "--", color="gray")
axes[1, 1].set_xlabel("false positive rate"); axes[1, 1].set_ylabel("true positive rate")
axes[1, 1].set_title("Test ROC overlay"); axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_ROOT / "lth_vs_resized_efficientnet.png", dpi=120)
plt.show()

# Confidence histogram
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prod_test_pred["prob_nits"], bins=30, alpha=0.55, label="production", color="crimson")
if best_by_val is not None:
    p = OUT_ROOT / best_by_val["init"] / f"{best_by_val['tag']}_test_predictions.csv"
    if p.exists():
        bp = pd.read_csv(p)
        ax.hist(bp["prob_nits"], bins=30, alpha=0.55,
                label=f"{best_by_val['init']}/{best_by_val['tag']}", color="#4C72B0")
ax.set_xlabel("P(nits)"); ax.set_ylabel("images")
ax.set_title("Test confidence distribution"); ax.legend()
plt.tight_layout()
plt.savefig(OUT_ROOT / "lth_confidence_hist.png", dpi=120)
plt.show()

print(f"Production EfficientNetB0: test F1={prod_test['f1']:.4f} | "
      f"AUC={prod_test['auc']:.4f} | comb detect={prod_sq['detect_pct']:.1f}% | "
      f"comb false={prod_sq['false_pct']:.1f}% | "
      f"TFLite dynamic={prod_tflite_dyn:.2f} MB")

if best_by_val is None:
    print("No ticket met the validation-F1 stopping criterion.")
else:
    print(f"Best ticket by validation F1: {best_by_val['init']}/{best_by_val['tag']} "
          f"(sparsity={best_by_val['actual_sparsity']:.3f})")
    print(f"  validation F1={best_by_val['val_f1']:.4f}")
    print(f"  test F1={best_by_val['test_f1']:.4f} "
          f"(delta vs production {best_by_val['f1_delta_vs_production']:+.4f})")
    print(f"  test AUC={best_by_val['test_auc']:.4f} "
          f"(delta {best_by_val['auc_delta_vs_production']:+.4f})")
    print(f"  comb detect={best_by_val['comb_detect_pct']:.1f}% | "
          f"false={best_by_val['comb_false_pct']:.1f}%")
    print(f"  effective size={best_by_val['effective_mb']:.2f} MB "
          f"({best_by_val['effective_size_reduction_pct']:.1f}% smaller)")
    print(f"  TFLite dynamic={best_by_val['tflite_dyn_mb']:.2f} MB "
          f"({best_by_val['tflite_size_reduction_pct']:.1f}% smaller)")

    viable = (best_by_val["test_f1"] >= prod_test["f1"] - 0.01
              and best_by_val["comb_detect_pct"] >= 95.0)
    if viable:
        print("Verdict: the ticket is a viable smaller replacement for the production model.")
    else:
        print("Verdict: the ticket is smaller but does not match the production model's "
              "accuracy/OOD behaviour.")

if best_by_test is not None:
    print(f"Best ticket by held-out test F1: {best_by_test['init']}/{best_by_test['tag']} "
          f"(test F1={best_by_test['test_f1']:.4f}, sparsity={best_by_test['actual_sparsity']:.3f})")


## 10. Interpretation notes

* `effective_mb` is the nonzero-parameter size (`nonzero_params × 4 bytes`). It is the
  LTH-relevant size.
* A dense `.keras` file still stores zeros, so `keras_mb` may not shrink even when sparsity
  is high.
* `tflite_dyn_mb` is the actual deployable compressed size. It is measured for the
  baseline and the best validation/test tickets only, because TFLite does not compress
  unstructured zeros and converting every round leaks memory. Dynamic-range quantization
  is independent of LTH, so it also shrinks the unpruned production model.
* `sparse_mb` is the `.npz` index+value size. At low sparsity the index overhead can make
  it larger than the dense `.keras`; it only wins at high sparsity.
* Pruning stops on **validation** F1. Test and comb metrics are reported afterward, which
  keeps the held-out test set out of the pruning decision.
* The random-init run is the faithful LTH setup. The pretrained-init run is an LTH-style
  variant that starts from ImageNet weights, so it may retain accuracy more easily.
* Per-layer sparsity files (`ticket_XX_layer_sparsity.csv`) show whether depthwise or
  squeeze-excite layers were pruned disproportionately.
